## Use a light Qwen from huggingface locally

### Download (only when needed) and load the LLM

In [ ]:
from download_model import *




import datetime
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download

# Define a lightweight LLM repository ID
# repo_id = "Qwen/Qwen2.5-1.5B-Instruct"
# # Model download
# model_file = download_huggingface_model(repo_id)
# print("Model path:", model_file)
# local_dir = os.path.dirname(model_file)


repo_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Download the ENTIRE repository snapshot (includes config, tokenizer, and weights)
# local_dir = snapshot_download(repo_id=repo_id)
local_dir = download_huggingface_model(repo_id=repo_id)
print("Model directory path:", local_dir)

# 2. Load the tokenizer and model from the local directory
print("Loading model into memory...")
tokenizer = AutoTokenizer.from_pretrained(local_dir, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    local_dir, 
    local_files_only=True,
    device_map="auto"
)





### Ask and Get the reply from the LLM

In [ ]:
# 3. Formulate the question (incorporating current real-world context if helpful)
# prompt = "How many days in February 2024? Please provide a brief explanation." #OK
prompt = "Is this a grammatially correct sentence: 'Money no enough'? Please explain why or why not." #OK

# 4. Tokenize and generate response
messages = [{"role": "system", "content": "You are a bird."},
            {"role": "user", "content": prompt}]
text_input = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

inputs = tokenizer([text_input], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=100,# decide how long the answer can be
        temperature=0.1
    )

# 5. Decode and print the output
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("\n--- Answer ---")
print(response.strip())